# 01 · Kronos path sampling + baseline eval (Colab T4)

1. Runtime → Change runtime type → **T4 GPU**.
2. Upload `kronos-agent.zip` to `MyDrive/kronos-agent/` (the setup cell unpacks it to `code/`).
3. Run all. Every stage is resumable: after a disconnect, re-run from the top and it skips finished origins.

Outputs on Drive: `data/` (cached OHLCV), `paths/<tag>/part-*.parquet` (sampled paths), `reports/<tag>/` (metrics).

In [ ]:
!nvidia-smi -L

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/kronos-agent'
CODE = f'{ROOT}/code'
KRONOS = '/content/Kronos'

# First run: unpack kronos-agent.zip (uploaded to MyDrive/kronos-agent/) into ROOT/code.
# To update code later, upload the new zip and set UPDATE = True once.
import os, shutil, zipfile
UPDATE = False
ZIP = f'{ROOT}/kronos-agent.zip'
if UPDATE or not os.path.exists(f'{CODE}/pyproject.toml'):
    assert os.path.exists(ZIP), f'upload kronos-agent.zip to {ROOT}/ first'
    shutil.rmtree(CODE, ignore_errors=True)
    zipfile.ZipFile(ZIP).extractall(CODE)
print(sorted(os.listdir(CODE)))

In [ ]:
!test -d $KRONOS || git clone -q https://github.com/shiyu-coder/Kronos $KRONOS
!git -C $KRONOS rev-parse HEAD
%pip install -q einops safetensors ccxt yfinance
%pip install -q -e "$CODE" --no-deps
import sys  # editable install only applies to new processes; make it importable in this kernel too
if CODE not in sys.path:
    sys.path.insert(0, CODE)

## Smoke test (3 origins) — check speed and output before the full run

In [ ]:
!cd $CODE && python scripts/run_sampling.py --config configs/btc_1h.yaml --root $ROOT --kronos-path $KRONOS --limit 3

## Full runs (~1–2 h total on T4; safe to interrupt and resume)

In [ ]:
!cd $CODE && python scripts/run_sampling.py --config configs/btc_1h.yaml --root $ROOT --kronos-path $KRONOS

In [ ]:
!cd $CODE && python scripts/run_sampling.py --config configs/spy_1d.yaml --root $ROOT --kronos-path $KRONOS

## Evaluation: Kronos vs Gaussian / bootstrap / random walk, pre- vs post-cutoff

In [ ]:
!cd $CODE && python scripts/run_eval.py --config configs/btc_1h.yaml --root $ROOT
!cd $CODE && python scripts/run_eval.py --config configs/spy_1d.yaml --root $ROOT

## Phase 2: conformal calibration + trade/abstain policy (CPU-only, minutes)

In [ ]:
!cd $CODE && python scripts/run_phase2.py --config configs/btc_1h.yaml --root $ROOT
!cd $CODE && python scripts/run_phase2.py --config configs/spy_1d.yaml --root $ROOT

## Diagnostics: PIT histogram (calibration) + one fan chart

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from kronos_agent.config import load_config, run_tag
from kronos_agent.data import load_data
from kronos_agent.store import ResultStore
from kronos_agent.evaluate import build_eval_frame
from kronos_agent.pipeline import decode_paths
from kronos_agent import metrics as M

cfg = load_config(f'{CODE}/configs/btc_1h.yaml')
df = load_data(cfg, f'{ROOT}/data')
tag = run_tag(cfg)
sdf = ResultStore(f'{ROOT}/paths/{tag}').load()
ev = build_eval_frame(sdf, df, cfg['lookback'], seed=cfg['seed'])

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for period, g in ev.groupby('period'):
    p = M.pit(np.stack(g['kronos']), g['r_true'].to_numpy())
    ax[0].hist(p, bins=20, range=(0, 1), alpha=0.5, density=True, label=period)
ax[0].axhline(1, color='k', lw=1); ax[0].set_title('Kronos PIT (flat = calibrated)'); ax[0].legend()

rec = sdf.iloc[-1]
paths = decode_paths(rec)
i, h = rec['row'], rec['pred_len']
hist = df['close'].iloc[i - 72:i]
fut = df.index[i:i + h]
ax[1].plot(hist.index, hist.values, color='k')
lo, mid, hi = np.quantile(paths, [0.05, 0.5, 0.95], axis=0)
ax[1].fill_between(fut, lo, hi, alpha=0.3, label='Kronos 90%')
ax[1].plot(fut, mid, label='median'); ax[1].plot(fut, df['close'].iloc[i:i + h], 'k--', label='realized')
ax[1].set_title(str(rec['origin_ts'])); ax[1].legend(); plt.tight_layout()